### Wave E — Unit type & booking path

Unit type from `flats`→`seriestypes`, else `unitselectiondetails` series. `flatintrestedsofar` is empty in this window (verified).

| Sheet | Yellow box |
|-------|------------|
| `UnitType_Interest_Volume` | #3 |
| `UnitType_Interest_vs_Booking` | #4 |
| `Direct_vs_NonDirect` | #5 |



In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path("/Users/lekhanigam/data/leads_sv_closure_reports")
sys.path.insert(0, str(ROOT))
from common import START_TS, END_TS, CLOSURE_STAGE, q, write_excel

DIRECT = {"Walk In","Channel Partner","Executive Referral","Phone","Customer Referral","Lead Referral","Self Generated Lead","Event"}

leads = q(f"""
WITH flat_u AS (
  SELECT CAST(f.lead AS varchar) lead_oid,
         CAST(st.name AS varchar) unit_type,
         1 AS pri,
         ROW_NUMBER() OVER (PARTITION BY CAST(f.lead AS varchar) ORDER BY f.updatedat DESC) rn
  FROM flats f
  LEFT JOIN seriestypes st ON CAST(st._id AS varchar)=CAST(f.seriestype AS varchar)
  WHERE f.lead IS NOT NULL
),
sel_u AS (
  SELECT CAST(h.lead AS varchar) lead_oid,
         json_extract_scalar(CAST(h.unitselectiondetails AS varchar), '$.differentUnitPreferences.series[0]') unit_type,
         2 AS pri,
         ROW_NUMBER() OVER (PARTITION BY CAST(h.lead AS varchar) ORDER BY h.createdat DESC) rn
  FROM leadstagehistories h
  WHERE h.createdat >= {START_TS} AND h.unitselectiondetails IS NOT NULL
),
unit AS (
  SELECT lead_oid, unit_type, pri FROM flat_u WHERE rn=1 AND unit_type IS NOT NULL AND TRIM(unit_type)<>''
  UNION ALL
  SELECT lead_oid, unit_type, pri FROM sel_u WHERE rn=1 AND unit_type IS NOT NULL AND TRIM(unit_type)<>''
),
unit_one AS (
  SELECT lead_oid, unit_type
  FROM (
    SELECT lead_oid, unit_type, ROW_NUMBER() OVER (PARTITION BY lead_oid ORDER BY pri) rn
    FROM unit
  ) z WHERE rn=1
)
SELECT CAST(l._id AS varchar) lead_oid,
       date_format(l.createdat, '%Y-%m') month_key,
       date_format(l.createdat, '%M') month_name,
       TRIM(CAST(ls.name AS varchar)) source,
       u.unit_type
FROM leads l
LEFT JOIN leadsources ls ON CAST(ls._id AS varchar)=CAST(l.bornsource AS varchar)
LEFT JOIN unit_one u ON u.lead_oid=CAST(l._id AS varchar)
WHERE l.createdat >= {START_TS} AND l.createdat < {END_TS}
""")
closed = set(q(f"""
SELECT CAST(lead AS varchar) lead_oid FROM leadstagehistories
WHERE CAST(stage AS varchar)='{CLOSURE_STAGE}' AND createdat >= {START_TS} AND createdat < {END_TS}
GROUP BY 1
""")["lead_oid"])
leads["is_closed"] = leads["lead_oid"].isin(closed).astype(int)
leads["booking_path"] = np.where(leads["source"].isin(DIRECT), "Direct", "Non-Direct")
u = leads[leads["unit_type"].notna() & (leads["unit_type"].astype(str).str.len() > 0)]
vs = u.groupby(["unit_type"], as_index=False).agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")})
vs["Conversion rate %"] = np.where(vs["Lead count"]>0, (100*vs["Sales closure count"]/vs["Lead count"]).round(2), 0)
write_excel(ROOT / "UnitType_Booking_Path_Analysis.xlsx", {
    "UnitType_Interest_Volume": u.groupby(["unit_type"], as_index=False).agg(**{"Lead count":("lead_oid","nunique")}).sort_values("Lead count", ascending=False),
    "UnitType_Interest_vs_Booking": vs.sort_values("Sales closure count", ascending=False),
    "Direct_vs_NonDirect": (
        leads.groupby(["booking_path","month_name","month_key"], as_index=False)
        .agg(**{"Lead count":("lead_oid","nunique"),"Sales closure count":("is_closed","sum")})
        .sort_values("month_key").drop(columns=["month_key"])
        .rename(columns={"booking_path": "Direct vs Non-Direct"})
    ),
})
print("with unit", len(u), "of", len(leads))

